# 03 — Creative Fatigue Analysis

Goal: understand the lifecycle of creative fatigue — when it starts, how CTR/CVR decay, and what a typical fatigue curve looks like.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA = Path('../')

cs = pd.read_csv(DATA / 'creative_summary.csv')
daily = pd.read_csv(DATA / 'creative_daily_country_os_stats.csv', parse_dates=['date'])

status_colors = {'top_performer': '#2ecc71', 'stable': '#3498db',
                 'fatigued': '#e67e22', 'underperformer': '#e74c3c'}
status_order = ['top_performer', 'stable', 'fatigued', 'underperformer']

print('Loaded. Fatigued creatives:', (cs['creative_status'] == 'fatigued').sum())

## 1. When Does Fatigue Hit? (fatigue_day distribution)

In [ ]:
fatigued = cs[cs['creative_status'] == 'fatigued'].copy()
print(f'Fatigued creatives with fatigue_day populated: {fatigued["fatigue_day"].notna().sum()}')
print(f'fatigue_day stats:\n{fatigued["fatigue_day"].describe().round(1)}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(fatigued['fatigue_day'].dropna(), bins=30, color='#e67e22', edgecolor='white', alpha=0.85)
ax.axvline(fatigued['fatigue_day'].median(), color='black', linestyle='--',
           linewidth=2, label=f'Median = {fatigued["fatigue_day"].median():.0f} days')
ax.set_xlabel('Day Since Launch (when fatigue began)')
ax.set_ylabel('Number of Creatives')
ax.set_title('Distribution of Fatigue Onset Day', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 2. CTR and CVR Decay by Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, label in [
    (axes[0], 'ctr_decay_pct', 'CTR Decay % (last 7d vs first 7d)'),
    (axes[1], 'cvr_decay_pct', 'CVR Decay % (last 7d vs first 7d)')
]:
    for status in status_order:
        vals = cs[cs['creative_status'] == status][col].dropna()
        # Clip extreme outliers for readability
        vals = vals.clip(-200, 200)
        ax.hist(vals, bins=40, alpha=0.55, label=status, color=status_colors[status], edgecolor='none')
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='no change')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Decay % (negative = decline)')
    ax.legend(fontsize=8)

plt.suptitle('CTR and CVR Decay Distributions by Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. First 7d vs Last 7d CTR Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for status in status_order:
    sub = cs[cs['creative_status'] == status].dropna(subset=['first_7d_ctr', 'last_7d_ctr'])
    ax.scatter(sub['first_7d_ctr'], sub['last_7d_ctr'],
               alpha=0.5, s=25, label=status, color=status_colors[status])

max_val = cs[['first_7d_ctr', 'last_7d_ctr']].max().max()
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, label='no change (diagonal)')
ax.set_xlabel('First 7-day CTR')
ax.set_ylabel('Last 7-day CTR')
ax.set_title('First 7d vs Last 7d CTR — Below the Diagonal = Declined', fontweight='bold')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## 4. Average Fatigue Curve — CTR over days_since_launch

In [ ]:
# Compute CTR at daily level
daily_agg = daily.groupby(['creative_id', 'days_since_launch']).agg(
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum')
).reset_index()
daily_agg['ctr'] = daily_agg['clicks'] / daily_agg['impressions'].replace(0, np.nan)

# Merge status
daily_agg = daily_agg.merge(cs[['creative_id', 'creative_status']], on='creative_id', how='left')

# Aggregate by status and days_since_launch (limit to first 60 days)
curve = daily_agg[daily_agg['days_since_launch'] <= 60].groupby(
    ['creative_status', 'days_since_launch']
)['ctr'].median().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
for status in status_order:
    sub = curve[curve['creative_status'] == status]
    ax.plot(sub['days_since_launch'], sub['ctr'],
            label=status, color=status_colors[status], linewidth=2)

ax.set_xlabel('Days Since Creative Launch')
ax.set_ylabel('Median Daily CTR')
ax.set_title('Average CTR Lifecycle by Creative Status (first 60 days)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Individual Fatigue Examples (5 fatigued creatives)

In [ ]:
# Pick 5 fatigued creatives with clear fatigue signals
sample_fatigued = fatigued.dropna(subset=['fatigue_day']).nlargest(5, 'ctr_decay_pct')['creative_id'].tolist()
# Also 3 stable for comparison
sample_stable = cs[cs['creative_status'] == 'stable'].nlargest(3, 'total_impressions')['creative_id'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=False)
axes = axes.flatten()

all_sample = [(cid, 'fatigued') for cid in sample_fatigued] + [(cid, 'stable') for cid in sample_stable]

for i, (cid, status) in enumerate(all_sample):
    sub = daily_agg[daily_agg['creative_id'] == cid].sort_values('days_since_launch')
    rolling_ctr = sub.set_index('days_since_launch')['ctr'].rolling(3, min_periods=1).mean()
    axes[i].plot(rolling_ctr.index, rolling_ctr.values, color=status_colors[status], linewidth=2)
    
    if status == 'fatigued':
        fd = cs.loc[cs['creative_id'] == cid, 'fatigue_day'].values[0]
        if not np.isnan(fd):
            axes[i].axvline(fd, color='red', linestyle='--', linewidth=1.5, label=f'Fatigue day {int(fd)}')
            axes[i].legend(fontsize=7)
    
    axes[i].set_title(f'Creative {cid} ({status})', fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Days Since Launch', fontsize=8)
    axes[i].set_ylabel('CTR (3d rolling)', fontsize=8)

plt.suptitle('CTR Trajectories: Fatigued vs Stable Creatives', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. What Predicts Early Fatigue? (fatigue_day correlations)

In [ ]:
corr_cols = ['fatigue_day', 'total_days_active', 'total_spend_usd',
             'total_impressions', 'first_7d_ctr', 'peak_rolling_ctr_5',
             'novelty_score', 'motion_score', 'clutter_score']

fatigue_corr = fatigued[corr_cols].corr()['fatigue_day'].drop('fatigue_day').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in fatigue_corr.values]
ax.barh(fatigue_corr.index, fatigue_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Pearson Correlation with fatigue_day\n(positive = later fatigue; negative = earlier fatigue)', fontweight='bold')
ax.set_xlabel('Correlation with fatigue_day')
plt.tight_layout()
plt.show()

print(fatigue_corr.round(3))